In [1]:
import os 
import logging

import hashlib
import hmac
import requests
import time
import json
from typing import Optional, Dict, Any, List, Union
from enum import Enum
from datetime import datetime
from dotenv import load_dotenv

import time
import logging
from typing import Dict, List, Optional, Union
from delta_rest_client import DeltaRestClient, OrderType

load_dotenv()

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class OrderType(Enum):
    LIMIT = "limit_order"
    MARKET = "market_order"
    STOP_MARKET = "stop_market_order"
    STOP_LIMIT = "stop_limit_order"
    BRACKET = "bracket_order"

class OrderSide(Enum):
    BUY = "buy"
    SELL = "sell"

class TimeInForce(Enum):
    GTC = "gtc"  # Good Till Cancelled
    IOC = "ioc"  # Immediate or Cancel
    FOK = "fok"  # Fill or Kill

class OrderState(Enum):
    OPEN = "open"
    PENDING = "pending"
    CLOSED = "closed"
    CANCELLED = "cancelled"
    PARTIALLY_FILLED = "partially_filled"
    FILLED = "filled"


class Broker:
    def __init__(self):
        # Initialize broker state if needed
        pass

    # def place_order(self, symbol, quantity, price, side, ordertype):
    #     """
    #     Place an order.
    #     :param symbol: Trading pair (e.g., 'BTCUSD')
    #     :param quantity: Amount to trade
    #     :param price: Order price
    #     :param side: 'buy' or 'sell'
    #     """
    #     raise NotImplementedError("This method should be implemented by subclasses")

    # def cancel_order(self, order_id):
    #     """
    #     Cancel an existing order.
    #     :param order_id: Unique identifier of the order
    #     """
    #     raise NotImplementedError("This method should be implemented by subclasses")


class DeltaExchange(Broker):
    def __init__(self, testnet: bool = False):
        super().__init__()
        self.session = requests.Session()
        self.api_key = os.getenv("API_KEY")
        self.api_secret = os.getenv("API_SECRET")
        self.allproducts = None
        print(self.api_key)

        if not self.api_key or not self.api_secret:
            raise ValueError("API_KEY and API_SECRET must be set")

        if testnet:
            self.base_url = "https://cdn-ind.testnet.deltaex.org"
            logger.info("Using TESTNET environment")
        else:
            self.base_url = "https://api.india.delta.exchange"
            logger.info("Using PRODUCTION environment")

                    # Initialize Delta REST client
        self.client = DeltaRestClient(
            base_url= self.base_url,
            api_key=self.api_key,
            api_secret=self.api_secret
        )

    def _load_products(self):
        """Load and cache product information"""
        if self.allproducts is None:
            try:
                self.allproducts = {}
                products = self.client.get_products()
                if products.get('success'):
                    for product in products.get('result', []):
                        symbol = product.get('symbol')
                        product_id = product.get('id')
                        self.allproducts[symbol] = product_id
                        self.allproducts[product_id] = symbol

                logger.info(f"Loaded {len(self.allproducts)} products")
            except Exception as e:
                logger.error(f"Failed to load products: {e}")

    def get_product_id(self, symbol: str) -> Optional[int]:
        """Get product ID for a given symbol"""
        return self.allproducts.get(symbol)
    
    def get_symbol(self, product_id: int) -> Optional[str]:
        """Get symbol for a given product ID"""
        return self.allproducts.get(product_id)
        
    
    def place_order(self, product_symbol: str, side: str, size: int, 
                   order_type: str = "LIMIT", limit_price: str = None,
                   time_in_force: str = None, reduce_only: bool = False,
                   client_order_id: str = None) -> Dict:
        """
        Place a new order using the delta_rest_client
        
        Args:
            product_symbol: Trading symbol (e.g., "BTCUSD")
            side: "buy" or "sell"
            size: Order size (integer, no fractional values)
            order_type: "LIMIT" or "MARKET"
            limit_price: Price for limit orders
            time_in_force: Order time in force
            reduce_only: Only close positions if True
            client_order_id: Custom order ID
            
        Returns:
            Order response
        """
        try:
            product_id = self.get_product_id(product_symbol)
            if not product_id:
                return {"success": False, "error": f"Product {product_symbol} not found"}
            
            # Convert string order type to OrderType enum
            if order_type.upper() == "LIMIT":
                order_type_enum = OrderType.LIMIT
            elif order_type.upper() == "MARKET":
                order_type_enum = OrderType.MARKET
            else:
                return {"success": False, "error": f"Unsupported order type: {order_type}"}
            
            # Convert side to proper format
            trade_type = side.lower()
            
            logger.info(f"Placing {side} order for {size} {product_symbol} (ID: {product_id})")
            
            # Use the specific method signature you provided
            response = self.client.place_order(
                product_id=product_id,
                qty=size,
                tradetype=trade_type,
                limit_price=limit_price,
                time_in_force=time_in_force,
                order_type=order_type_enum
            )
            
            return response
            
        except Exception as e:
            logger.error(f"Error placing order: {e}")
            return {"success": False, "error": str(e)}

    def get_balance(self, asset_id = 'BTC'):
        return self.client.get_balances(asset_id=asset_id)

broker = DeltaExchange()
balance = broker.get_balance()
assets = broker.client.get_assets()
print(type(assets))
# logger.info(F"Trading Amount : {balance} rs.")
# minbalance = 0
# if balance <= minbalance:
#     logger.critical(f"Balance is Less than {minbalance}")
#     raise(f"Balance is Less than {minbalance}")
#start Trading 
  
        

2025-11-27 17:12:15,570 - INFO - Using PRODUCTION environment


50eL5nk80t0rXaXSPcUu3IvOZYMKPq
<class 'list'>


In [ ]:
for asset in assets:
    if asset['symbol'] == 'USD':
        print(asset)
    # balance = broker.client.get_balances(asset_id=asset['id'])
    # if balance:
        # print(asset)
        # print(balance)

{'base_withdrawal_fee': '0.000000000000000000', 'id': 14, 'interest_credit': False, 'interest_slabs': None, 'kyc_deposit_limit': '0.000000000000000000', 'kyc_withdrawal_limit': '0.000000000000000000', 'min_withdrawal_amount': '0.000000000000000000', 'minimum_precision': 2, 'name': 'US Dollar', 'networks': [{'allowed_deposit_groups': None, 'base_withdrawal_fee': '0', 'deposit_status': 'enabled', 'memo_required': False, 'min_deposit_amount': '0.000000000000000000', 'min_withdrawal_amount': '0.000000000000000000', 'minimum_deposit_confirmations': 1, 'network': 'INR_BANKING', 'variable_withdrawal_fee': '0', 'withdrawal_status': 'enabled'}], 'precision': 8, 'sort_priority': 3, 'symbol': 'USD', 'variable_withdrawal_fee': '0.000000000000000000'}


In [11]:
for i in broker.client.get_live_orders():
    print(i)

{'product_symbol': 'BTCUSD', 'mmp': 'disabled', 'bracket_trail_amount': None, 'average_fill_price': None, 'unfilled_size': 1, 'meta_data': {'ip': '27.107.167.114', 'otc': False, 'source': 'api'}, 'state': 'open', 'quote_size': None, 'paid_commission': '0', 'bracket_take_profit_limit_price': None, 'id': 1058574502, 'bracket_stop_loss_limit_price': None, 'cancellation_reason': None, 'client_order_id': None, 'stop_price': None, 'side': 'buy', 'bracket_order': None, 'stop_order_type': None, 'product': {'contract_type': 'perpetual_futures', 'contract_unit_currency': 'BTC', 'contract_value': '0.001', 'id': 27, 'notional_type': 'vanilla', 'quoting_asset': {'minimum_precision': 2, 'precision': 8, 'symbol': 'USD'}, 'settling_asset': {'minimum_precision': 2, 'precision': 8, 'symbol': 'USD'}, 'spot_index': {'symbol': '.DEXBTUSD'}, 'symbol': 'BTCUSD', 'tick_size': '0.5', 'underlying_asset': {'minimum_precision': 4, 'precision': 8, 'symbol': 'BTC'}}, 'user_id': 88133346, 'reduce_only': False, 'trai

In [3]:
broker.client.get_position(27)

{'entry_price': '91390.00000000', 'size': 1}

In [21]:
allorders_org = broker.client.order_history(query={}, page_size=50)

In [22]:
allorders = allorders_org['result']

In [ ]:
symbol = 'BTCUSD'
for order in allorders:
    if order['product_symbol'] == symbol and order['order_type'] == 'limit_order':
        print(order)
        print(order['id'])

In [27]:
broker.client.get_position(27)

{'entry_price': '91390.00000000', 'size': 1}

In [ ]:
orders = [{'average_fill_price': None, 'bracket_order': True, 'bracket_stop_loss_limit_price': None, 'bracket_stop_loss_price': None, 'bracket_take_profit_limit_price': None, 'bracket_take_profit_price': None, 'bracket_trail_amount': None, 'cancellation_reason': None, 'client_order_id': None, 'commission': '0', 'created_at': '2025-11-28T09:51:36.715575Z', 'id': 1059756005, 'limit_price': None, 'meta_data': {'ip': '2409:40c0:32:2dbd:7ca7:cff:fec7:8842', 'source': 'mobile_app_android'}, 'mmp': 'disabled', 'order_type': 'market_order', 'paid_commission': '0', 'product_id': 27, 'product_symbol': 'BTCUSD', 'quote_size': None, 'reduce_only': True, 'side': 'sell', 'size': 1, 'state': 'pending', 'stop_order_type': 'stop_loss_order', 'stop_price': '90000', 'stop_trigger_method': 'mark_price', 'time_in_force': 'gtc', 'trail_amount': None, 'unfilled_size': 1, 'updated_at': '2025-11-28T09:51:36.715575Z', 'user_id': 88133346}, {'average_fill_price': None, 'bracket_order': True, 'bracket_stop_loss_limit_price': None, 'bracket_stop_loss_price': None, 'bracket_take_profit_limit_price': None, 'bracket_take_profit_price': None, 'bracket_trail_amount': None, 'cancellation_reason': None, 'client_order_id': None, 'commission': '0', 'created_at': '2025-11-28T09:51:36.716215Z', 'id': 1059756006, 'limit_price': None, 'meta_data': {'ip': '2409:40c0:32:2dbd:7ca7:cff:fec7:8842', 'source': 'mobile_app_android'}, 'mmp': 'disabled', 'order_type': 'market_order', 'paid_commission': '0', 'product_id': 27, 'product_symbol': 'BTCUSD', 'quote_size': None, 'reduce_only': True, 'side': 'sell', 'size': 1, 'state': 'pending', 'stop_order_type': 'take_profit_order', 'stop_price': '93000', 'stop_trigger_method': 'mark_price', 'time_in_force': 'gtc', 'trail_amount': None, 'unfilled_size': 1, 'updated_at': '2025-11-28T09:51:36.716215Z', 'user_id': 88133346}]
for order in orders:
    
    print(order)

{'average_fill_price': None, 'bracket_order': True, 'bracket_stop_loss_limit_price': None, 'bracket_stop_loss_price': None, 'bracket_take_profit_limit_price': None, 'bracket_take_profit_price': None, 'bracket_trail_amount': None, 'cancellation_reason': None, 'client_order_id': None, 'commission': '0', 'created_at': '2025-11-28T09:51:36.715575Z', 'id': 1059756005, 'limit_price': None, 'meta_data': {'ip': '2409:40c0:32:2dbd:7ca7:cff:fec7:8842', 'source': 'mobile_app_android'}, 'mmp': 'disabled', 'order_type': 'market_order', 'paid_commission': '0', 'product_id': 27, 'product_symbol': 'BTCUSD', 'quote_size': None, 'reduce_only': True, 'side': 'sell', 'size': 1, 'state': 'pending', 'stop_order_type': 'stop_loss_order', 'stop_price': '90000', 'stop_trigger_method': 'mark_price', 'time_in_force': 'gtc', 'trail_amount': None, 'unfilled_size': 1, 'updated_at': '2025-11-28T09:51:36.715575Z', 'user_id': 88133346}
{'average_fill_price': None, 'bracket_order': True, 'bracket_stop_loss_limit_price'

In [ ]:
import websocket
import json

WEBSOCKET_URL = "wss://socket.india.delta.exchange"

def on_message(ws, message):
    print(json.loads(message))

def on_open(ws):
    ws.send(json.dumps({
        "type": "subscribe",
        "payload": {
            "channels": [{
                "name": "v2/ticker",
                "symbols": ["BTCUSD"]
            }]
        }
    }))

ws = websocket.WebSocketApp(WEBSOCKET_URL, on_message=on_message)
ws.on_open = on_open
ws.run_forever()


In [ ]:
{'mark_basis': '-0.00042967',
'turnover': 812673745.4740059,
 'size': 8911410, 
 'timestamp': 1764329555217238,
   'open': 91309.0,
     'volume': 8911.410000008094,
       'close': 91259.0, 
       'type':'v2/ticker',
         'oi_value_usd': '44234454.8230',
           'oi_contracts': '484510',
             'high': 91998.5,
               'oi_value_symbol': 'BTC',
                 'symbol': 'BTCUSD',
                   'low': 90419.5,
                     'turnover_symbol': 'USD',
                       'underlying_asset_symbol': 'BTC',
                         'price_band': {'lower_limit': '86727.49205807', 'upper_limit': '95856.70174839'},
                           'oi_change_usd_6h': '-443783.1600',
                             'spot_price': '91296.9',
                               'funding_rate': '0.010000000000000002',
                                 'oi': '484.5100',
                                   'ltp_change_24h': '-0.0548',
                                     'description': 'Bitcoin Perpetual',
                                       'tags': ['layer_1'],
                                         'sort_priority': 1,
                                           'mark_change_24h': '-0.1331',
                                             'contract_type': 'perpetual_futures',
                                               'turnover_usd': 812673745.4740059,
                                                 'product_trading_status': 'operational',
                                                   'product_id': 27,
                                                     'mark_price': '91258.07208955',
                                                       'quotes': {'ask_iv': None, 'ask_size': '4292', 'best_ask': '91260', 'best_ask_mm': '91260', 'best_bid': '91259', 'best_bid_mm': '91259', 'bid_iv': None, 'bid_size': '1319', 'impact_mid_price': None, 'mark_iv': '-0.48200243'}, 'leverage': 200, 'contract_value': '0.001000000000000000', 'tick_size': '0.500000000000000000', 'oi_value': '484.5100', 'greeks': None, 'time': '2025-11-27T11:31:14.203575308Z'}